# 1-Year Cumulative Incidence per AE (Figure 2B)

Bar chart of 1-year cumulative incidence (%) for 6 irAE types, sorted by incidence. 95% CIs computed but not shown on the plot — exported to CSV instead.

In [ ]:
%matplotlib inline
import re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None

TOXICITY_COLUMNS = ["pneumonitis", "adrenal_insufficiency", "liver_toxicity",
                    "colitis", "hyperthyroidism", "hypothyroidism"]
TOXICITY_DISPLAY = {
    "pneumonitis": "Pneumonitis", "adrenal_insufficiency": "Adrenal Insufficiency",
    "liver_toxicity": "Liver Toxicity", "colitis": "Colitis",
    "hyperthyroidism": "Hyperthyroidism", "hypothyroidism": "Hypothyroidism",
}
AE_COLORS = {
    "pneumonitis": "#D55E00", "adrenal_insufficiency": "#009E73",
    "liver_toxicity": "#0072B2", "colitis": "#E69F00",
    "hyperthyroidism": "#56B4E9", "hypothyroidism": "#CC79A7",
}

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'main'))
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONE_PATH = os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(BACKBONE_PATH, low_memory=False)
batch_df = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv'), encoding="latin-1", low_memory=False)

covars["mrn"] = covars["mrn"].apply(standardize_mrn)
batch_df["mrn"] = batch_df["mrn"].apply(standardize_mrn)
covars = covars[covars["mrn"].notna()].copy()
batch_df = batch_df[batch_df["mrn"].notna()].copy()

covars["lot_start"] = pd.to_datetime(covars["lot_start"], errors="coerce")
covars["lot"] = pd.to_numeric(covars["lot"], errors="coerce")
covars["censor_days"] = pd.to_numeric(covars["t_cutoff_lot"], errors="coerce")
covars = covars.sort_values(["mrn", "lot"])

batch_df["window_start"] = pd.to_datetime(batch_df["window_start"], errors="coerce")
batch_df["window_end"] = pd.to_datetime(batch_df["window_end"], errors="coerce")
batch_df = batch_df.dropna(subset=["window_start", "window_end"])

print(f"Covars: {covars.shape}")
print(f"Batch: {batch_df.shape}, {batch_df['mrn'].nunique():,} patients")

In [ ]:
# Build line1: first LOT per patient, using the pre-computed t_cutoff_lot (censor_days) directly
covars_valid = covars[covars["lot_start"].notna()].copy()
line1 = (covars_valid.sort_values(["mrn", "lot"])
         .groupby("mrn").first().reset_index()[["mrn", "lot_start", "censor_days"]]
         .rename(columns={"lot_start": "line1_start"}))
line1 = line1[np.isfinite(line1["censor_days"]) & (line1["censor_days"] > 0)].copy()
print(f"Line 1 patients with valid censoring: {len(line1):,}")

In [ ]:
# Merge batch with line1
batch_merged = batch_df.merge(line1, on="mrn", how="inner")
batch_merged["days_from_start"] = (batch_merged["window_start"] - batch_merged["line1_start"]).dt.days
batch_merged = batch_merged[
    (batch_merged["days_from_start"] >= 0) &
    (batch_merged["days_from_start"] <= batch_merged["censor_days"])
].copy()
print(f"Batch records after merge & filtering: {len(batch_merged):,}")

In [ ]:
# Compute 1-year cumulative incidence with 95% CI for each toxicity
T_MONTHS = 12.0
results = []
for tox in TOXICITY_COLUMNS:
    if tox not in batch_merged.columns:
        continue
    first_ae = batch_merged[batch_merged[tox] == 1].copy()
    first_ae = first_ae[first_ae["days_from_start"] >= 0]
    first_ae = first_ae.groupby("mrn")["days_from_start"].min().reset_index().rename(
        columns={"days_from_start": "time"})
    first_ae["event"] = 1
    surv = line1[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
    surv["event"] = surv["event"].fillna(0).astype(int)
    surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
    surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
    surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
    surv = surv[surv["time"] > 0].copy()
    surv["time_months"] = surv["time"] / 30.44
    kmf = KaplanMeierFitter()
    kmf.fit(surv["time_months"], surv["event"])
    ci_at = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    ci_tbl = kmf.confidence_interval_survival_function_
    idx = max(0, min(np.searchsorted(kmf.survival_function_.index, T_MONTHS, side="right") - 1,
                     len(ci_tbl) - 1))
    ci_lo = (1 - ci_tbl.iloc[idx, 1]) * 100
    ci_hi = (1 - ci_tbl.iloc[idx, 0]) * 100
    results.append((tox, ci_at, ci_lo, ci_hi))
    print(f"  {TOXICITY_DISPLAY[tox]}: {ci_at:.1f}% ({ci_lo:.1f}-{ci_hi:.1f})")

results.sort(key=lambda x: x[1], reverse=True)

In [ ]:
# Export CI table to CSV
ci_df = pd.DataFrame(results, columns=["toxicity_key", "pct_1yr", "ci_lower", "ci_upper"])
ci_df["toxicity"] = ci_df["toxicity_key"].map(TOXICITY_DISPLAY)
ci_df = ci_df[["toxicity", "pct_1yr", "ci_lower", "ci_upper"]]
os.makedirs(RESULTS_DIR, exist_ok=True)
ci_df.to_csv(os.path.join(RESULTS_DIR, 'One_Year_Cumulative_Incidence_2B_CI.csv'), index=False)
ci_df

In [ ]:
los = np.array([r[2] for r in results])
his = np.array([r[3] for r in results])

In [ ]:
# Plot
labels = [TOXICITY_DISPLAY[t] for t, *_ in results]
bar_colors = [AE_COLORS[t] for t, *_ in results]
pcts = np.array([r[1] for r in results])

fig, ax = plt.subplots(figsize=(4.0, 2.8))
bars = ax.bar(range(len(pcts)), pcts, width=0.6, color=bar_colors, edgecolor="none",
              yerr=[pcts - los, his - pcts], capsize=3, ecolor="black",
              error_kw={"linewidth": 0.8})
for i in range(len(pcts)):
    ax.text(i, his[i] + max(his) * 0.03,
            f'{pcts[i]:.1f}%',
            ha='center', va='bottom', fontsize=5, color='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=6, rotation=30, ha="right", rotation_mode="anchor")
ax.set_ylabel("1-year cumulative incidence (%)", fontsize=7)
ax.tick_params(axis="y", labelsize=6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, max(his) * 1.15)
fig.subplots_adjust(left=0.15, right=0.97, top=0.95, bottom=0.22)

In [ ]:
# Save
os.makedirs(RESULTS_DIR, exist_ok=True)
with PdfPages(os.path.join(RESULTS_DIR, 'One_Year_Cumulative_Incidence_2B.pdf')) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print("Saved: ../results/main/One_Year_Cumulative_Incidence_2B.pdf")